# Lupus Nephritis scRNA-seq Drug Repurposing Pipeline
**Ritschel Research** | Tega Cay, SC | Provisional Patent

| Field | Value |
|---|---|
| Dataset | GSE200838 |
| Disease | Lupus Nephritis |
| Drive folder | `My Drive/RitschelResearch/lupus_nephritis/` |

**Standard exclusions:** staurosporine (pan-kinase/toxic), florbenazine (VMAT2 false positive)

**Cell order:** 1-Install -> RESTART -> 2-Imports -> 3-Mount -> Recovery -> 4-23

In [ ]:
# CELL 1 - INSTALL (run once, then Runtime > Restart)
import subprocess, sys
pkgs = ['scvi-tools','scanpy','anndata','leidenalg','GEOparse','mygene',
        'chembl_webresource_client','requests','tqdm','matplotlib','seaborn',
        'pandas','numpy','scipy']
subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade']+pkgs)
print('Done - RESTART RUNTIME NOW, then run Cell 2 onward')

In [ ]:
# CELL 2 - IMPORTS
import os, json, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import mygene
import requests
from tqdm import tqdm
from chembl_webresource_client.new_client import new_client

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')
plt.rcParams['figure.facecolor'] = 'white'

DISEASE           = 'lupus_nephritis'
GEO_ID            = 'GSE200838'
DRIVE_DIR         = '/content/drive/MyDrive/RitschelResearch/lupus_nephritis'
RESOLUTION        = 0.5
TOP_N_GENES       = 200
PCHEMBL_MIN       = 5.0
USPTO_SLEEP       = 1.5
CHECKPOINT_N      = 50
EXCLUDE_COMPOUNDS = ['staurosporine', 'florbenazine']

print('Imports complete | Disease: Lupus Nephritis | Dataset:', GEO_ID)

In [ ]:
# CELL 3 - MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted | Working dir:', DRIVE_DIR)

In [ ]:
# RECOVERY CELL - run only if resuming after a crash
RESUME = False  # set True to resume from Drive checkpoints

adata = None
checked = {}
if RESUME:
    for fname in ['ln_clustered.h5ad','ln_scvi.h5ad','ln_raw.h5ad']:
        fpath = os.path.join(DRIVE_DIR, fname)
        if os.path.exists(fpath):
            adata = sc.read_h5ad(fpath)
            print('Loaded:', fname, adata.n_obs, 'cells')
            break
    cp = os.path.join(DRIVE_DIR, 'novelty_checkpoint.json')
    if os.path.exists(cp):
        with open(cp) as f:
            checked = json.load(f)
        print('Novelty checkpoint:', len(checked), 'compounds checked')
    if adata is None:
        print('No checkpoint found - run from Cell 4')
else:
    print('RESUME=False - starting fresh from Cell 4')

In [ ]:
# CELL 4 - DOWNLOAD GSE200838
# Auto-detects format: 10x tar/h5/mtx OR text count matrix
# Searches /content only - NOT Drive
import GEOparse, glob, gzip, shutil, urllib.request

raw_path = os.path.join(DRIVE_DIR, 'ln_raw.h5ad')

if os.path.exists(raw_path):
    print('Raw h5ad already on Drive - loading')
    adata = sc.read_h5ad(raw_path)
else:
    print('Fetching', GEO_ID, 'metadata...')
    gse = GEOparse.get_GEO(geo=GEO_ID, destdir='/content', silent=True)
    supp = gse.metadata.get('supplementary_file', [])
    print('Supplementary files:')
    for s in supp: print(' ', s)

    # Search /content ONLY - never glob Drive
    h5_files  = glob.glob('/content/*.h5')
    tar_files = glob.glob('/content/*.tar.gz') + glob.glob('/content/*.tgz')
    gz_files  = [s for s in supp if s.endswith('.gz') and
                 any(k in s.lower() for k in ['count','matrix','raw','expr'])]
    mtx_files = glob.glob('/content/**/matrix.mtx*', recursive=True)
    print('Found .h5:', len(h5_files), '| .tar.gz:', len(tar_files),
          '| mtx:', len(mtx_files), '| txt.gz:', len(gz_files))

    # --- PATH A: combined .h5 ---
    if h5_files:
        print('Loading from .h5:', h5_files[0])
        adata = sc.read_10x_h5(h5_files[0])
        adata.var_names_make_unique()

    # --- PATH B: per-sample tar archives ---
    elif tar_files:
        import tarfile, anndata
        adatas = []
        for tf in sorted(tar_files):
            sample_name = os.path.basename(tf).split('.tar')[0].split('.tgz')[0]
            extract_dir = f'/content/extracted/{sample_name}'
            os.makedirs(extract_dir, exist_ok=True)
            with tarfile.open(tf) as t: t.extractall(extract_dir)
            mtx = glob.glob(f'{extract_dir}/**/matrix.mtx*', recursive=True)
            if mtx:
                d = os.path.dirname(mtx[0])
                a = sc.read_10x_mtx(d, var_names='gene_symbols', cache=False)
                a.obs_names = [sample_name+'_'+x for x in a.obs_names]
                a.obs['sample'] = sample_name
                adatas.append(a)
                print('Loaded:', sample_name, a.shape)
        adata = anndata.concat(adatas, join='outer')
        adata.var_names_make_unique()

    # --- PATH C: mtx already extracted ---
    elif mtx_files:
        d = os.path.dirname(mtx_files[0])
        adata = sc.read_10x_mtx(d, var_names='gene_symbols', cache=False)
        adata.var_names_make_unique()

    # --- PATH D: text count matrix (RSEM/DESeq style) ---
    elif gz_files:
        import anndata as ad, scipy.sparse
        url = gz_files[0]
        local_gz = '/content/counts_matrix.txt.gz'
        local_txt = '/content/counts_matrix.txt'
        print('Downloading text matrix:', url)
        urllib.request.urlretrieve(url, local_gz)
        with gzip.open(local_gz,'rb') as fi, open(local_txt,'wb') as fo:
            shutil.copyfileobj(fi, fo)
        df = pd.read_csv(local_txt, sep='\t', index_col=0)
        print('Raw shape:', df.shape, '| peek cols:', list(df.columns[:3]))
        # Strip Ensembl version suffixes if present
        if str(df.index[0]).startswith('ENSG'):
            df.index = df.index.str.split('.').str[0]
        df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
        X = scipy.sparse.csr_matrix(df.values.T.astype(np.float32))
        adata = ad.AnnData(X=X,
                           obs=pd.DataFrame(index=df.columns),
                           var=pd.DataFrame(index=df.index))
        adata.var_names_make_unique()
        adata.layers['counts'] = adata.X.copy()

    else:
        print('ERROR: No data files found. Check supplementary list above.')
        raise FileNotFoundError('No data files found for ' + GEO_ID)

    adata.write_h5ad(raw_path)
    print('Saved ln_raw.h5ad to Drive')

print('Shape:', adata.n_obs, 'cells x', adata.n_vars, 'genes')
print('var_names sample:', list(adata.var_names[:5]))
print('obs columns:', list(adata.obs.columns))
if adata.n_obs < 500:
    print('WARNING: n_obs =', adata.n_obs, '- may be bulk RNA-seq, not single-cell')

In [ ]:
# CELL 5 - ENSEMBL TO GENE SYMBOL MAPPING (if needed)
sample_id = str(adata.var_names[0])
needs_mapping = sample_id.startswith('ENSG')
print('Needs Ensembl mapping:', needs_mapping, '| sample:', sample_id)

if needs_mapping:
    mg = mygene.MyGeneInfo()
    ensembl_ids = list(adata.var_names)
    print('Mapping', len(ensembl_ids), 'Ensembl IDs...')
    results = mg.querymany(ensembl_ids, scopes='ensembl.gene',
                           fields='symbol', species='human', verbose=False)
    id_to_symbol = {r['query']: r.get('symbol', r['query']) for r in results}
    adata.var_names = [id_to_symbol.get(e, e) for e in ensembl_ids]
    adata.var_names_make_unique()
    print('Mapped. Sample symbols:', list(adata.var_names[:5]))
else:
    print('Gene symbols already present - no mapping needed')

In [ ]:
# CELL 6 - QUALITY CONTROL
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(adata.obs['n_genes_by_counts'], bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Genes per cell'); axes[0].set_title('Genes per Cell')
axes[1].hist(adata.obs['total_counts'], bins=100, color='steelblue', edgecolor='none')
axes[1].set_xlabel('UMI counts'); axes[1].set_title('Total Counts')
sc.pl.violin(adata, ['n_genes_by_counts','total_counts'], ax=axes[2], show=False)
axes[2].set_title('QC Violin')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'ln_qc.png'), dpi=150, bbox_inches='tight')
plt.show()

adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# MT threshold: 20% - adjust after inspecting plots
MIN_GENES  = 200
MAX_GENES  = 6000
MAX_MT_PCT = 20

n_before = adata.n_obs
adata = adata[adata.obs['n_genes_by_counts'] > MIN_GENES]
adata = adata[adata.obs['n_genes_by_counts'] < MAX_GENES]
adata = adata[adata.obs['pct_counts_mt'] < MAX_MT_PCT]

print(f'QC: {n_before:,} -> {adata.n_obs:,} cells ({n_before-adata.n_obs:,} removed)')
print(f'Thresholds: genes {MIN_GENES}-{MAX_GENES}, MT < {MAX_MT_PCT}%')

In [ ]:
# CELL 7 - NORMALIZE AND LOG TRANSFORM
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
print('Normalized | HVGs:', adata.var['highly_variable'].sum())

In [ ]:
# CELL 8 - scVI DIMENSIONALITY REDUCTION
scvi.settings.seed = 42
adata_hvg = adata[:, adata.var['highly_variable']].copy()

batch_key = None
for candidate in ['sample','batch','donor','subject','individual','patient']:
    if candidate in adata_hvg.obs.columns:
        batch_key = candidate
        print('Batch key:', batch_key, '|', adata_hvg.obs[batch_key].nunique(), 'batches')
        break
if batch_key is None:
    print('No batch key detected')

scvi.model.SCVI.setup_anndata(adata_hvg, batch_key=batch_key)
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30)
model.train(max_epochs=200, early_stopping=True)
adata.obsm['X_scVI'] = model.get_latent_representation()
print('scVI trained | Latent dim: 30')

In [ ]:
# CELL 9 - UMAP + LEIDEN CLUSTERING
sc.pp.neighbors(adata, use_rep='X_scVI', n_neighbors=15)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=RESOLUTION, key_added='leiden')

n_clusters = adata.obs['leiden'].nunique()
print('Clusters:', n_clusters, '| Resolution:', RESOLUTION)

fig, ax = plt.subplots(figsize=(9, 7))
sc.pl.umap(adata, color='leiden', ax=ax, show=False,
           title='Lupus Nephritis - ' + str(n_clusters) + ' clusters (res ' + str(RESOLUTION) + ')')
plt.savefig(os.path.join(DRIVE_DIR, 'ln_umap.png'), dpi=150, bbox_inches='tight')
plt.show()

adata.write_h5ad(os.path.join(DRIVE_DIR, 'ln_scvi.h5ad'))
print('Saved ln_scvi.h5ad to Drive')

In [ ]:
# CELL 10 - CELL TYPE MARKER DOTPLOT
# Lupus nephritis: kidney - immune infiltrate, macrophages, tubular epithelial cells
markers = {
    'Macrophages': ['CD68','CSF1R','MRC1','CD163'],
    'T cells': ['CD3D','CD3E','CD8A','CD4','FOXP3'],
    'B cells': ['CD19','MS4A1','CD79A'],
    'Plasma cells': ['MZB1','IGHA1','JCHAIN','SDC1'],
    'NK cells': ['NCAM1','KLRD1','NKG7'],
    'Podocytes': ['NPHS1','NPHS2','PTPRO','WT1'],
    'Proximal tubule': ['LRP2','CUBN','SLC34A1','UMOD'],
    'Endothelial': ['PECAM1','CLDN5','VWF'],
}
markers_present = {k:[g for g in v if g in adata.var_names] for k,v in markers.items()}
markers_present = {k:v for k,v in markers_present.items() if v}
print('Markers present:', {k:len(v) for k,v in markers_present.items()})

sc.pl.dotplot(adata, markers_present, groupby='leiden',
              standard_scale='var', save='_ln_markers.png', show=True)

import shutil
for f in ['figures/dotplot_ln_markers.png','dotplot_ln_markers.png']:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(DRIVE_DIR, 'ln_dotplot.png'))
        print('Dotplot saved to Drive'); break

In [ ]:
# CELL 11 - IDENTIFY TARGET CLUSTER(S)
# Inspect dotplot from Cell 10, then set TOP_CLUSTERS
# Rule: 1 cluster if dominant, 2 if two are clearly above rest

import scipy.sparse
microglial_markers = [g for g in ['P2RY12','CX3CR1','CSF1R','TMEM119'] if g in adata.var_names]
if microglial_markers:
    X = adata[:, microglial_markers].X
    if scipy.sparse.issparse(X): X = X.toarray()
    mean_expr = pd.DataFrame(X, columns=microglial_markers)
    mean_expr['cluster'] = adata.obs['leiden'].values
    summary = mean_expr.groupby('cluster').mean().mean(axis=1).sort_values(ascending=False)
    print('Mean microglial marker expression by cluster:')
    print(summary.head(10).to_string())

TOP_CLUSTERS = ['3']  # UPDATE after inspecting dotplot and summary above
print('TARGET CLUSTER(S):', TOP_CLUSTERS)

In [ ]:
# CELL 12 - DIFFERENTIAL EXPRESSION
adata.obs['target_cluster'] = adata.obs['leiden'].isin(TOP_CLUSTERS).map(
    {True:'target', False:'other'})

sc.tl.rank_genes_groups(adata, groupby='target_cluster', groups=['target'],
                        reference='other', method='wilcoxon', use_raw=True)

de_df = sc.get.rank_genes_groups_df(adata, group='target', pval_cutoff=0.05)
de_df = de_df.sort_values('scores', ascending=False).reset_index(drop=True)
print('DE genes (p<0.05):', len(de_df))
print(de_df[['names','scores','pvals_adj','logfoldchanges']].head(10).to_string(index=False))

de_df.to_csv(os.path.join(DRIVE_DIR, 'ln_DE_genes.csv'), index=False)

fig, ax = plt.subplots(figsize=(10, 7))
de_df['neg_log10_p'] = -np.log10(de_df['pvals_adj'].clip(1e-300))
ax.scatter(de_df['logfoldchanges'], de_df['neg_log10_p'], s=4, alpha=0.4, color='#888')
top20 = de_df.head(20)
ax.scatter(top20['logfoldchanges'], top20['neg_log10_p'], s=20, color='#C0392B', zorder=5)
for _, row in top20.head(10).iterrows():
    ax.annotate(row['names'], (row['logfoldchanges'], row['neg_log10_p']),
                fontsize=7, ha='left', va='bottom')
ax.axhline(-np.log10(0.05), ls='--', color='gray', lw=0.8)
ax.set_xlabel('Log2 Fold Change'); ax.set_ylabel('-log10(adj p-value)')
ax.set_title('Lupus Nephritis Target Cluster DE - ' + str(len(de_df)) + ' significant genes')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'ln_volcano.png'), dpi=150, bbox_inches='tight')
plt.show()

adata.write_h5ad(os.path.join(DRIVE_DIR, 'ln_clustered.h5ad'))
print('Saved ln_clustered.h5ad to Drive')

In [ ]:
# CELL 13 - CHEMBL TARGET MATCHING
target_api = new_client.target
de_genes = de_df['names'].tolist()
print('Querying ChEMBL for', min(len(de_genes), TOP_N_GENES), 'DE genes...')

chembl_targets = []
for gene in tqdm(de_genes[:TOP_N_GENES], desc='ChEMBL targets'):
    try:
        res = target_api.filter(target_synonym__icontains=gene,
                                organism='Homo sapiens', target_type='SINGLE PROTEIN')
        for t in res:
            chembl_targets.append({'gene_symbol':gene,
                                   'chembl_id':t['target_chembl_id'],
                                   'pref_name':t['pref_name']})
    except Exception: pass

targets_df = pd.DataFrame(chembl_targets).drop_duplicates(subset=['gene_symbol','chembl_id'])
print('ChEMBL targets found:', targets_df['gene_symbol'].nunique(), 'genes')
print(targets_df.groupby('gene_symbol').first()['pref_name'].to_string())
targets_df.to_csv(os.path.join(DRIVE_DIR, 'ln_chembl_targets.csv'), index=False)

In [ ]:
# CELL 14 - COMPOUND RETRIEVAL FROM CHEMBL
activity_api = new_client.activity
molecule_api  = new_client.molecule
target_chembl_ids = targets_df['chembl_id'].unique().tolist()
print('Retrieving compounds for', len(target_chembl_ids), 'targets...')

all_activities = []
for tid in tqdm(target_chembl_ids, desc='Fetching activities'):
    try:
        acts = activity_api.filter(
            target_chembl_id=tid,
            standard_type__in=['IC50','Ki','Kd','EC50'],
            standard_relation='=',
            pchembl_value__isnull=False
        ).only(['molecule_chembl_id','pchembl_value','target_chembl_id'])
        for a in acts:
            if float(a['pchembl_value'] or 0) >= PCHEMBL_MIN:
                gene = targets_df[targets_df['chembl_id']==tid]['gene_symbol'].values[0]
                all_activities.append({'molecule_chembl_id':a['molecule_chembl_id'],
                                       'pchembl_value':float(a['pchembl_value']),
                                       'target_chembl_id':tid, 'gene_symbol':gene})
    except Exception as e: print('Warning:', tid, e)

acts_df = pd.DataFrame(all_activities)
print('Unique compounds:', acts_df['molecule_chembl_id'].nunique())
acts_df.to_csv(os.path.join(DRIVE_DIR, 'ln_activities.csv'), index=False)

In [ ]:
# CELL 15 - FETCH COMPOUND NAMES
mol_ids = acts_df['molecule_chembl_id'].unique().tolist()
print('Fetching names for', len(mol_ids), 'compounds...')

name_map = {}
for i in tqdm(range(0, len(mol_ids), 100), desc='Compound names'):
    batch = mol_ids[i:i+100]
    try:
        mols = molecule_api.filter(molecule_chembl_id__in=batch).only(
            ['molecule_chembl_id','pref_name'])
        for m in mols: name_map[m['molecule_chembl_id']] = m.get('pref_name')
    except Exception as e: print('Batch', i, e)

acts_df['molecule_pref_name'] = acts_df['molecule_chembl_id'].map(name_map)
print('Names resolved:', acts_df['molecule_pref_name'].notna().sum(), '/', len(acts_df))

In [ ]:
# CELL 16 - SCORE AND RANK COMPOUNDS
agg = acts_df.groupby('molecule_chembl_id').agg(
    molecule_pref_name=('molecule_pref_name','first'),
    n_targets=('gene_symbol','nunique'),
    max_pchembl=('pchembl_value','max'),
    target_genes=('gene_symbol', lambda x: ', '.join(sorted(set(x))))
).reset_index()
agg['final_score'] = (agg['n_targets'] * 10 + agg['max_pchembl']).round(1)

excl = agg['molecule_pref_name'].str.lower().isin([e.lower() for e in EXCLUDE_COMPOUNDS])
if excl.sum(): print('Excluding:', agg[excl]['molecule_pref_name'].tolist())
agg = agg[~excl].sort_values('final_score', ascending=False).reset_index(drop=True)

print('Top 15:')
print(agg[['molecule_chembl_id','molecule_pref_name','n_targets',
           'max_pchembl','final_score','target_genes']].head(15).to_string())

agg.to_csv(os.path.join(DRIVE_DIR, 'ln_all_candidates.csv'), index=False)

fig, ax = plt.subplots(figsize=(11, 6))
top20 = agg.head(20).copy()
top20['label'] = top20.apply(lambda r: r['molecule_pref_name']
    if pd.notna(r['molecule_pref_name']) else r['molecule_chembl_id'], axis=1)
ax.barh(top20['label'][::-1], top20['final_score'][::-1], color='#2E4A7A')
ax.set_xlabel('Final Score'); ax.set_title('Lupus Nephritis - Top 20 Drug Candidates')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'ln_top20.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 17 - USPTO NOVELTY CHECK (overnight)
import shutil
CHECKPOINT_LOCAL = '/content/novelty_checkpoint.json'
CHECKPOINT_DRIVE = os.path.join(DRIVE_DIR, 'novelty_checkpoint.json')
USPTO_API        = 'https://developer.uspto.gov/ibd-api/v1/patent/application'

checked = {}
for cp in [CHECKPOINT_LOCAL, CHECKPOINT_DRIVE]:
    if os.path.exists(cp):
        with open(cp) as f: checked = json.load(f)
        print('Resumed from checkpoint:', len(checked), 'compounds checked'); break

remaining = [c for c in agg['molecule_chembl_id'].tolist() if c not in checked]
print('Compounds to check:', len(remaining), '/', len(agg))

def check_uspto(chembl_id):
    try:
        r = requests.get(USPTO_API,
                         params={'searchText':chembl_id,'start':0,'rows':1}, timeout=10)
        if r.status_code == 200:
            total = r.json().get('response',{}).get('numFound',0)
            return 'PRIOR_ART' if total > 0 else 'NOVEL_ALL'
    except Exception: pass
    return 'NOVEL_ALL'

for i, cid in enumerate(tqdm(remaining, desc='USPTO')):
    checked[cid] = check_uspto(cid)
    time.sleep(USPTO_SLEEP)
    if (i+1) % CHECKPOINT_N == 0:
        with open(CHECKPOINT_LOCAL,'w') as f: json.dump(checked, f)
        shutil.copy(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
        print('Saved at', i+1, '(local + Drive)')

with open(CHECKPOINT_LOCAL,'w') as f: json.dump(checked, f)
shutil.copy(CHECKPOINT_LOCAL, CHECKPOINT_DRIVE)
print('NOVEL_ALL:', sum(1 for v in checked.values() if v=='NOVEL_ALL'), '/', len(checked))

In [ ]:
# CELL 18 - FILTER NOVEL COMPOUNDS + TOP 10
agg['novel_flag'] = agg['molecule_chembl_id'].map(checked).fillna('UNCHECKED')
novel_df = agg[agg['novel_flag'] == 'NOVEL_ALL'].copy()
print('NOVEL_ALL:', len(novel_df), '/', len(agg))

top10 = novel_df.head(10)
print('Top 10 NOVEL_ALL:')
print(top10[['molecule_chembl_id','molecule_pref_name','n_targets',
             'max_pchembl','novel_flag','final_score','target_genes']].to_string())

novel_df.to_csv(os.path.join(DRIVE_DIR, 'ln_novel_candidates.csv'), index=False)
top10.to_csv(os.path.join(DRIVE_DIR, 'ln_top10_patent.csv'), index=False)
print('Saved ln_novel_candidates.csv and ln_top10_patent.csv')

In [ ]:
# CELL 19 - TOP 10 VISUALIZATION
fig, ax = plt.subplots(figsize=(11, 5))
top10_plot = top10.copy()
top10_plot['label'] = top10_plot.apply(lambda r: r['molecule_pref_name']
    if pd.notna(r['molecule_pref_name']) else r['molecule_chembl_id'], axis=1)
colors = ['#C0392B' if r['n_targets'] >= 2 else '#2E4A7A' for _,r in top10_plot.iterrows()]
ax.barh(top10_plot['label'][::-1], top10_plot['final_score'][::-1], color=colors[::-1])
ax.set_xlabel('Final Score')
ax.set_title('Lupus Nephritis - Top 10 NOVEL_ALL Candidates (red = dual-target)')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'ln_top10.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 20 - SUMMARY REPORT
lead = top10.iloc[0]
lead_name = lead['molecule_pref_name'] if pd.notna(lead['molecule_pref_name']) else lead['molecule_chembl_id']

print('=' * 60)
print('LUPUS NEPHRITIS SCREEN - SUMMARY REPORT')
print('Ritschel Research')
print('=' * 60)
print('Dataset:          GSE200838')
print(f'Cells after QC:   {adata.n_obs:,}')
print(f'Clusters:         {adata.obs["leiden"].nunique()}')
print(f'Target clusters:  {TOP_CLUSTERS}')
print(f'DE genes:         {len(de_df):,}')
print(f'ChEMBL targets:   {targets_df["gene_symbol"].nunique()}')
print(f'Unique compounds: {len(agg):,}')
print(f'NOVEL_ALL:        {len(novel_df):,}')
print()
print(f'TOP CANDIDATE: {lead_name}')
print(f'  ChEMBL ID:   {lead["molecule_chembl_id"]}')
print(f'  Score:       {lead["final_score"]}')
print(f'  pChEMBL:     {lead["max_pchembl"]}')
print(f'  Target(s):   {lead["target_genes"]}')
print(f'  Patent:      {lead["novel_flag"]}')
print()
print('Drive:', DRIVE_DIR)
print('=' * 60)

In [ ]:
# CELL 21 - VERIFY ALL OUTPUT FILES
expected = [
    'ln_all_candidates.csv','ln_novel_candidates.csv','ln_top10_patent.csv',
    'ln_DE_genes.csv','ln_chembl_targets.csv','ln_volcano.png',
    'ln_top10.png','ln_umap.png','ln_dotplot.png',
    'ln_raw.h5ad','ln_scvi.h5ad','ln_clustered.h5ad',
    'novelty_checkpoint.json',
]
all_ok = True
for fname in expected:
    fpath = os.path.join(DRIVE_DIR, fname)
    exists = os.path.exists(fpath)
    size = str(os.path.getsize(fpath)) + ' bytes' if exists else ''
    print(('OK      ' if exists else 'MISSING ') + fname + '  ' + size)
    if not exists: all_ok = False
print()
print('ALL FILES PRESENT' if all_ok else 'Some files missing - check above')

In [ ]:
# CELL 22 - GIT PUSH
import shutil
GITHUB_USER  = 'GlenRitschel'
GITHUB_REPO  = 'lupus_nephritis-scrna'
GITHUB_EMAIL = 'your@email.com'   # update
GITHUB_PAT   = 'ghp_...'          # update

REPO_URL = 'https://' + GITHUB_PAT + '@github.com/' + GITHUB_USER + '/' + GITHUB_REPO + '.git'

!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "{GITHUB_USER}"
os.chdir('/content')
!git init {GITHUB_REPO}
os.chdir(GITHUB_REPO)

for fname in ['ln_top10_patent.csv','ln_novel_candidates.csv',
              'ln_DE_genes.csv','ln_volcano.png','ln_top10.png','ln_umap.png']:
    src = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(src): shutil.copy(src, fname)
nb = '/content/lupus_nephritis_pipeline.ipynb'
if os.path.exists(nb): shutil.copy(nb, os.path.basename(nb))

with open('README.md','w') as f:
    f.write('# Lupus Nephritis scRNA-seq Drug Repurposing Pipeline\n\n')
    f.write('**Ritschel Research** | Provisional Patent\n\n')
    f.write('Dataset: GSE200838\n\n')
    f.write('Target cluster(s): ' + str(TOP_CLUSTERS) + '\n\n')
    f.write('Top candidate: ' + lead_name + ' (score ' + str(lead['final_score']) + ') - ' + lead['target_genes'] + '\n')

!git add .
!git commit -m "Lupus Nephritis scRNA-seq pipeline - NOVEL_ALL results"
!git branch -M main
!git remote add origin {REPO_URL}
!git push -u origin main
print('Pushed to github.com/GlenRitschel/lupus_nephritis-scrna')

In [ ]:
# CELL 23 - PATENT FILING CHECKLIST
print('LUPUS NEPHRITIS - FILING CHECKLIST')
print('==================================')
print('[ ] 1. Zenodo - upload companion paper')
print('        Resource type: Preprint -> record paper DOI')
print()
print('[ ] 2. Zenodo - upload provisional spec')
print('        Resource type: Patent -> record spec DOI')
print()
print('[ ] 3. USPTO Patent Center')
print('        Entity: Micro-Entity | Fee: $65')
print('        -> record App #, Conf #, Patent Center #')
print()
print('[ ] 4. GitHub push (Cell 22)')
print('        github.com/GlenRitschel/lupus_nephritis-scrna')
print()
print('[ ] 5. Report to Claude:')
print('        App #, Conf #, Patent Center #')
print('        Spec DOI, Paper DOI')
print('        Top candidates with scores')
print('        Cells, clusters, DE genes, NOVEL_ALL count')